# Solar Cycle Prediction: Implementation / 태양 주기 예측 구현

**Paper**: Petrovay, K. (2020), "Solar cycle prediction", *Living Reviews in Solar Physics*, **17**:2. DOI: [10.1007/s41116-020-0022-z](https://doi.org/10.1007/s41116-020-0022-z)

---

### English
This notebook reproduces the key predictive methods reviewed by Petrovay (2020):

1. **Polar field precursor method**: linear regression of cycle $n+1$ amplitude on the polar field measurement at the minimum of cycle $n$.
2. **Waldmeier effect**: the empirical anti-correlation between a cycle's rise time and its maximum amplitude.
3. **Autoregressive (AR) time-series model** applied to smoothed sunspot numbers.
4. **Simple kinematic dynamo (truncated oscillator) model** with nonlinear quenching (Duffing / van der Pol form).
5. **LSTM neural network toy example** for cycle amplitude prediction.
6. **Cycle 25 prediction visualization** combining ensemble forecasts with uncertainty.

### 한국어
이 노트북은 Petrovay(2020)의 리뷰에서 다룬 주요 예측 방법론을 재현한다:

1. **극자기장 선행지표 방법**: 주기 $n$의 극소기 극자기장에 대한 주기 $n+1$의 진폭 선형 회귀.
2. **Waldmeier 효과**: 주기의 상승 시간과 최대 진폭 사이의 경험적 역상관 관계.
3. **자기회귀(AR) 시계열 모델**을 평활화된 흑점 수에 적용.
4. **단순 운동학적 다이나모 (truncated oscillator) 모델**: 비선형 quenching (Duffing/van der Pol 형식).
5. **LSTM 신경망 장난감 예시**: 주기 진폭 예측.
6. **Cycle 25 예측 시각화**: 앙상블 예측과 불확실성을 결합.

## 0. Setup / 설정

### English
Import standard scientific Python libraries. The analysis only requires `numpy`, `matplotlib`, `scipy`, and (optionally) `sklearn` and `torch` for the ML examples.

### 한국어
표준 과학 계산 라이브러리를 import 한다. 분석에는 `numpy`, `matplotlib`, `scipy`가 필요하며, ML 예시에는 `sklearn`과 `torch`가 선택적으로 필요하다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import odeint

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
print("NumPy:", np.__version__)

## 1. Polar Field Precursor Method / 극자기장 선행지표 방법

### English
The polar field precursor method is based on the empirical correlation between the polar magnetic field strength at the cycle minimum and the amplitude of the *next* sunspot cycle. The physical basis is the Babcock-Leighton mechanism: the polar field at minimum seeds the next cycle's toroidal field.

We reproduce Table 1 from Petrovay (2020) and compute a linear fit:
$$R_{\max}^{(n+1)} = c_1 \cdot B_{\text{polar, min}}^{(n)}$$

### 한국어
극자기장 선행지표 방법은 주기 극소기의 극자기장 강도와 **다음** 흑점 주기의 진폭 사이의 경험적 상관에 기반한다. 물리적 근거는 Babcock-Leighton 메커니즘이다: 극소기의 극자기장이 다음 주기의 toroidal 장의 씨앗이 된다.

Petrovay (2020)의 Table 1을 재현하고 선형 fit을 계산한다.

In [ ]:
# Table 1 from Petrovay (2020): Polar field predictors and cycle amplitudes
# Cycles 21-24 with SSN v2 amplitudes

data = {
    "cycle":   [21,    22,    23,    24],
    "R_max":   [232.9, 212.5, 180.3, 116.4],     # peak SSN v2
    "WSO_min": [1.05,  1.28,  1.00,  0.52],      # Gauss, polar field at previous minimum
    "WSO_max": [1.07,  1.31,  1.13,  0.63],      # peak polar field
    "D_min":   [4.10,  3.98,  3.01,  1.33],      # axial dipole at minimum
    "D_max":   [4.19,  4.23,  3.96,  1.95],      # peak axial dipole
}

# Cycle 25 predictor values (late 2018, approximate - actual minimum was May 2019)
cycle25 = {"WSO_min": 0.72, "D_min": 1.93}

print("Observed cycle amplitudes (SSN v2):")
for i, c in enumerate(data["cycle"]):
    print(f"  Cycle {c}: R_max = {data['R_max'][i]:.1f}, B_polar(min) = {data['WSO_min'][i]:.2f} G")
print(f"\nCycle 25 predictor (Dec 2018): B_polar ~ {cycle25['WSO_min']:.2f} G")

In [ ]:
# Linear regression: R_max(cycle n+1) = c * B_polar(min of cycle n)
# Use previous-cycle minimum polar field vs current cycle amplitude
B = np.array(data["WSO_min"])
R = np.array(data["R_max"])

# Fit without intercept (through origin) per Petrovay's convention (Table 1)
c_fit = np.sum(B * R) / np.sum(B ** 2)
residuals = R - c_fit * B
scatter = np.sqrt(np.mean(residuals ** 2))

print(f"Fit coefficient: c = {c_fit:.1f}")
print(f"Residual scatter (RMS): {scatter:.1f}")

# Cycle 25 prediction using Cycle 24 minimum polar field
R25_pred = c_fit * cycle25["WSO_min"]
R25_lower = R25_pred - scatter
R25_upper = R25_pred + scatter
print(f"\nCycle 25 prediction: R_max = {R25_pred:.0f} +/- {scatter:.0f}")
print(f"  Range: {R25_lower:.0f} - {R25_upper:.0f}")

In [ ]:
# Visualize polar precursor regression
fig, ax = plt.subplots(figsize=(10, 6))

B_line = np.linspace(0, 1.5, 100)
R_line = c_fit * B_line

ax.fill_between(B_line, R_line - scatter, R_line + scatter,
                alpha=0.2, color="steelblue", label=f"+/- 1 sigma band")
ax.plot(B_line, R_line, "-", color="steelblue", linewidth=2, label=f"Linear fit: R = {c_fit:.1f} B")

# Data points
for i, c in enumerate(data["cycle"]):
    ax.plot(data["WSO_min"][i], data["R_max"][i], "o", markersize=12, color="darkorange")
    ax.annotate(f"Cycle {c}", (data["WSO_min"][i], data["R_max"][i]),
                xytext=(8, 8), textcoords="offset points", fontsize=11)

# Cycle 25 prediction
ax.errorbar(cycle25["WSO_min"], R25_pred, yerr=scatter, fmt="s", markersize=14,
            color="crimson", capsize=6, label=f"Cycle 25 forecast: {R25_pred:.0f} +/- {scatter:.0f}")
ax.annotate("Cycle 25", (cycle25["WSO_min"], R25_pred),
            xytext=(8, 8), textcoords="offset points", fontsize=11, color="crimson")

ax.set_xlabel("WSO polar field at cycle minimum [G] / 주기 극소기 극자기장")
ax.set_ylabel("Next cycle peak amplitude R_max [SSN v2] / 다음 주기 피크 진폭")
ax.set_title("Polar Field Precursor Method / 극자기장 선행지표 방법\n(Petrovay 2020, Table 1)")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1.5)
ax.set_ylim(0, 280)
plt.tight_layout()
plt.show()

print(f"\nCorrelation coefficient: r = {np.corrcoef(B, R)[0, 1]:.3f}")

## 2. Waldmeier Effect / Waldmeier 효과

### English
Waldmeier's empirical rule states that stronger cycles have **shorter rise times**. This creates an anti-correlation between rise time $(t_{\max} - t_{\min})$ and cycle amplitude.

Petrovay (2020) shows several functional forms:
- Original Waldmeier (Eq. 8): $\log R_{\max} = C_1 - C_2 (t_{\max} - t_{\min})$
- Kitiashvili-Kosovichev dynamo (Eq. 6): $R_{\max} = C_1 - C_2 (t_{\max} - t_{\min})$

We use synthetic data based on historical rise times and amplitudes for cycles 1-24.

### 한국어
Waldmeier의 경험 규칙: 강한 주기는 **상승 시간이 짧다**. 상승 시간 $(t_{\max} - t_{\min})$과 주기 진폭 사이에 역상관이 발생한다.

Petrovay (2020)는 여러 함수 형태를 제시한다:
- 원래 Waldmeier (식 8): $\log R_{\max} = C_1 - C_2 (t_{\max} - t_{\min})$
- Kitiashvili-Kosovichev 다이나모 (식 6): $R_{\max} = C_1 - C_2 (t_{\max} - t_{\min})$

역사적 상승 시간과 진폭을 기반으로 한 합성 데이터를 사용한다.

In [ ]:
# Historical rise times (years) and max amplitudes for cycles 1-24 (approximate, SSN v1 scale)
# Data compiled from SIDC records
cycles_all = np.arange(1, 25)
rise_times = np.array([6.3, 3.4, 3.3, 5.4, 4.7, 5.7, 6.3, 3.5, 4.7, 3.3,
                       3.2, 5.5, 5.3, 6.6, 4.0, 4.3, 4.0, 3.6, 3.3, 4.4,
                       3.3, 2.7, 5.3, 5.3])  # years
R_max_all = np.array([86.5, 115.8, 158.5, 141.2, 49.2, 48.7, 71.7, 146.9,
                      131.6, 97.9, 140.5, 74.6, 87.9, 64.2, 105.4, 78.1,
                      119.2, 151.8, 201.3, 110.6, 164.5, 158.5, 120.8, 81.8])

# Remove the anomalous Cycle 19 as Petrovay does
mask = cycles_all != 19
rt = rise_times[mask]
Rm = R_max_all[mask]

# Linear fit: R_max = a - b * rise_time
slope, intercept, r_value, p_value, std_err = stats.linregress(rt, Rm)

print(f"Waldmeier fit (excluding Cycle 19):")
print(f"  R_max = {intercept:.1f} - {-slope:.1f} * rise_time")
print(f"  Correlation: r = {r_value:.3f}, p = {p_value:.4f}")

# Log form (original Waldmeier, Eq. 8)
log_Rm = np.log10(Rm)
slope_log, intercept_log, r_log, _, _ = stats.linregress(rt, log_Rm)
print(f"\nLog Waldmeier (Eq. 8):")
print(f"  log(R_max) = {intercept_log:.3f} - {-slope_log:.3f} * rise_time")
print(f"  Correlation: r = {r_log:.3f}")

In [ ]:
# Visualize Waldmeier effect
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Linear form
ax = axes[0]
rt_line = np.linspace(2, 7, 100)
Rm_line = intercept + slope * rt_line
ax.plot(rt_line, Rm_line, "-", color="steelblue", linewidth=2,
        label=f"Fit: R = {intercept:.0f} - {-slope:.1f} t, r = {r_value:.2f}")
ax.scatter(rt, Rm, s=80, color="darkorange", edgecolor="black", zorder=3)
# Mark anomalous Cycle 19
ax.scatter([rise_times[18]], [R_max_all[18]], s=120, color="red", marker="X",
           edgecolor="black", zorder=3, label="Cycle 19 (outlier)")

for i, c in enumerate(cycles_all):
    ax.annotate(str(c), (rise_times[i], R_max_all[i]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)

ax.set_xlabel("Rise time (years) / 상승 시간 (년)")
ax.set_ylabel("Peak amplitude R_max / 피크 진폭")
ax.set_title("Waldmeier Effect (linear) / Waldmeier 효과 (선형)")
ax.legend()
ax.grid(True, alpha=0.3)

# Log form
ax = axes[1]
log_line = intercept_log + slope_log * rt_line
ax.plot(rt_line, log_line, "-", color="steelblue", linewidth=2,
        label=f"log(R) = {intercept_log:.2f} - {-slope_log:.2f} t")
ax.scatter(rt, np.log10(Rm), s=80, color="darkorange", edgecolor="black", zorder=3)
ax.set_xlabel("Rise time (years) / 상승 시간 (년)")
ax.set_ylabel("log10(R_max) / log10(피크 진폭)")
ax.set_title("Waldmeier Effect (log form, Eq. 8) / Waldmeier 효과 (로그 형식)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Autoregressive AR(2) Model / 자기회귀 AR(2) 모델

### English
Petrovay (§4.1) reviews AR and ARMA models applied to the SSN series. We fit an AR(2) model to a synthetic cycle-amplitude time series:

$$R_n = c_0 + c_1 R_{n-1} + c_2 R_{n-2} + \epsilon_n$$

Brajša et al. (2009) applied ARMA(6,6) to annual $R$ values, predicting Cycle 24 = 90 +/- 27 — within the right range (observed 116).

### 한국어
Petrovay(§4.1)는 SSN 시계열에 적용된 AR 및 ARMA 모델을 검토한다. 우리는 주기 진폭 합성 시계열에 AR(2) 모델을 적합한다:

$$R_n = c_0 + c_1 R_{n-1} + c_2 R_{n-2} + \epsilon_n$$

Brajša et al.(2009)는 연간 $R$ 값에 ARMA(6,6)를 적용하여 Cycle 24 = 90 +/- 27을 예측했다 — 올바른 범위 (실제 116).

In [ ]:
def fit_ar2(series):
    """Fit an AR(2) model to a 1D time series using least squares.

    Args:
        series: 1D numpy array of the time-series values.

    Returns:
        Tuple (c0, c1, c2, residuals) where coefficients satisfy
        R_n = c0 + c1 * R_{n-1} + c2 * R_{n-2}.
    """
    n = len(series)
    y = series[2:]
    X = np.column_stack([np.ones(n - 2), series[1:n - 1], series[0:n - 2]])
    coeffs, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    residuals = y - X @ coeffs
    return coeffs[0], coeffs[1], coeffs[2], residuals


def predict_ar2(series, n_future, c0, c1, c2, noise_std=0.0):
    """Forecast future values using a fitted AR(2) model.

    Args:
        series: 1D array with the last two points providing initial conditions.
        n_future: Number of future points to predict.
        c0, c1, c2: AR(2) coefficients.
        noise_std: Standard deviation of added Gaussian noise.

    Returns:
        1D array of forecast values (length n_future).
    """
    out = list(series[-2:])
    for _ in range(n_future):
        next_val = c0 + c1 * out[-1] + c2 * out[-2]
        if noise_std > 0:
            next_val += np.random.normal(0, noise_std)
        out.append(next_val)
    return np.array(out[2:])

# Fit AR(2) to cycle amplitudes 1-23 and predict 24, 25
train = R_max_all[:23]  # Cycles 1-23
c0, c1, c2, resid = fit_ar2(train)

print(f"AR(2) fit: R_n = {c0:.2f} + {c1:.3f} R_(n-1) + {c2:.3f} R_(n-2)")
print(f"Residual std: {np.std(resid):.2f}")

forecast = predict_ar2(train, 2, c0, c1, c2, noise_std=np.std(resid))
print(f"\nAR(2) forecast:")
print(f"  Cycle 24: {forecast[0]:.1f} (observed: {R_max_all[23]:.1f})")
print(f"  Cycle 25: {forecast[1]:.1f}")

In [ ]:
# Monte-Carlo ensemble AR(2) forecast for uncertainty
n_mc = 1000
forecasts = np.zeros((n_mc, 2))
for i in range(n_mc):
    forecasts[i] = predict_ar2(train, 2, c0, c1, c2, noise_std=np.std(resid))

print(f"AR(2) Monte-Carlo forecast (N={n_mc}):")
print(f"  Cycle 24: {np.mean(forecasts[:, 0]):.1f} +/- {np.std(forecasts[:, 0]):.1f}")
print(f"  Cycle 25: {np.mean(forecasts[:, 1]):.1f} +/- {np.std(forecasts[:, 1]):.1f}")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cycles_all[:23], train, "o-", color="steelblue", markersize=8, label="Observed (Cycles 1-23)")
ax.plot(cycles_all[23], R_max_all[23], "s", color="darkorange", markersize=12, label="Observed Cycle 24")
ax.errorbar([24, 25], np.mean(forecasts, axis=0), yerr=np.std(forecasts, axis=0),
            fmt="D", color="crimson", markersize=12, capsize=6, label="AR(2) forecast")
ax.set_xlabel("Cycle number / 주기 번호")
ax.set_ylabel("Peak amplitude R_max / 피크 진폭")
ax.set_title("AR(2) Time-Series Forecast / AR(2) 시계열 예측")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Kinematic Dynamo Oscillator Model / 운동학적 다이나모 진동자 모델

### English
Petrovay (§3.6) discusses truncated dynamo models that reduce the full $\alpha\Omega$ system to ordinary differential equations. One popular form is the van der Pol oscillator (Eq. 24):

$$\ddot{\xi} = -\xi + \mu (1 - \xi^2) \dot{\xi}$$

This self-excited oscillator with amplitude-dependent damping can reproduce key features of the solar cycle including asymmetric profiles and grand minima when stochastically forced.

### 한국어
Petrovay(§3.6)는 완전한 $\alpha\Omega$ 시스템을 ODE로 축소한 truncated 다이나모 모델을 논의한다. 인기 있는 형태 중 하나는 van der Pol 진동자(식 24):

$$\ddot{\xi} = -\xi + \mu (1 - \xi^2) \dot{\xi}$$

진폭 의존적 감쇠를 가진 이 자기 여진 진동자는 확률적 강제 하에서 비대칭 프로파일 및 grand minima를 포함하는 태양 주기의 주요 특징을 재현할 수 있다.

In [ ]:
def van_der_pol(state, t, mu, forcing_amp=0.0, forcing_freq=1.0, noise=None):
    """Van der Pol oscillator with optional stochastic forcing.

    Args:
        state: 2-element array [xi, xi_dot].
        t: Time (scalar).
        mu: Nonlinearity parameter (larger = more nonlinear, asymmetric).
        forcing_amp: Amplitude of sinusoidal forcing term.
        forcing_freq: Frequency of sinusoidal forcing.
        noise: Optional callable noise(t) returning scalar noise value.

    Returns:
        State derivative [xi_dot, xi_ddot].
    """
    xi, xi_dot = state
    ext = forcing_amp * np.sin(forcing_freq * t)
    if noise is not None:
        ext += noise(t)
    xi_ddot = -xi + mu * (1 - xi ** 2) * xi_dot + ext
    return [xi_dot, xi_ddot]


# Integrate van der Pol oscillator
t = np.linspace(0, 150, 3000)
mu = 1.5
state0 = [0.5, 0.0]

solution = odeint(van_der_pol, state0, t, args=(mu, 0.0, 1.0, None))

# Convert to ~SSN-like: |xi| -> rectified toroidal field proxy, then scale
R_dynamo = 100 * np.abs(solution[:, 0])

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(t, solution[:, 0], color="steelblue", linewidth=1.2)
axes[0].set_xlabel("Time (dimensionless) / 무차원 시간")
axes[0].set_ylabel("xi (toroidal field proxy)")
axes[0].set_title(f"van der Pol Oscillator Solution (mu = {mu}) / van der Pol 진동자 해")
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, R_dynamo, color="darkorange", linewidth=1.2)
axes[1].set_xlabel("Time / 시간")
axes[1].set_ylabel("|xi| x 100 (R proxy)")
axes[1].set_title("Rectified Signal (SSN-like proxy) / 정류된 신호")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Average cycle amplitude: {R_dynamo[len(R_dynamo)//2:].max():.1f}")

In [ ]:
# Simple Babcock-Leighton-like kinematic dynamo (truncated Eqs. 19-20)
def bl_dynamo(state, t, alpha, Omega_over_L, tau, noise=None):
    """Simple Babcock-Leighton truncated dynamo.

    Args:
        state: [A, B] where A is poloidal, B is toroidal.
        t: Time.
        alpha: Effective alpha parameter.
        Omega_over_L: Differential rotation shear parameter.
        tau: Decay timescale.
        noise: Optional additive noise on A.

    Returns:
        [A_dot, B_dot] state derivative.
    """
    A, B = state
    noise_val = noise(t) if noise is not None else 0.0
    A_dot = alpha * B - A / tau + noise_val
    B_dot = Omega_over_L * A - B / tau
    return [A_dot, B_dot]


t = np.linspace(0, 100, 2000)
alpha = 1.0
Omega_L = 1.2
tau = 2.0
D = alpha * Omega_L * tau ** 2 / 1.0  # dynamo number
print(f"Dynamo number D = {D:.2f}")

# Add stochastic forcing to simulate rogue AR fluctuations
def rogue_noise(t):
    return 0.3 * np.random.randn()

# Solve multiple realizations (ensemble)
n_ensemble = 20
fig, ax = plt.subplots(figsize=(12, 6))
for i in range(n_ensemble):
    np.random.seed(i)
    sol = odeint(bl_dynamo, [0.1, 0.05], t, args=(alpha, Omega_L, tau, rogue_noise))
    B_signal = np.abs(sol[:, 1])
    ax.plot(t, B_signal, alpha=0.4, linewidth=0.8)

ax.set_xlabel("Time / 시간")
ax.set_ylabel("|B| (toroidal field proxy)")
ax.set_title(f"Ensemble BL Dynamo with Stochastic Forcing / 확률적 강제 BL 다이나모 앙상블 (N={n_ensemble})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. LSTM Toy Example / LSTM 장난감 예시

### English
Petrovay (§4.3.4) discusses neural networks. We implement a minimal LSTM (Long Short-Term Memory) model using PyTorch to learn patterns in the cycle amplitude sequence.

**Note**: With only 24 cycles of data, this is a severely under-constrained problem. Real LSTM-based cycle prediction (e.g., Covas et al. 2019) uses the full butterfly diagram with thousands of data points. This toy example is for illustration only.

### 한국어
Petrovay(§4.3.4)는 신경망을 논의한다. PyTorch를 사용하여 주기 진폭 시퀀스의 패턴을 학습하는 최소한의 LSTM(Long Short-Term Memory) 모델을 구현한다.

**주의**: 24개 주기 데이터만으로는 심하게 과소제약된 문제다. 실제 LSTM 기반 주기 예측(예: Covas et al. 2019)은 수천 개 데이터 포인트를 가진 butterfly diagram 전체를 사용한다. 이 장난감 예시는 설명용이다.

In [ ]:
try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
except ImportError:
    TORCH_OK = False
    print("PyTorch not available; skipping LSTM example.")

if TORCH_OK:
    class CycleLSTM(nn.Module):
        """Minimal LSTM for solar cycle amplitude prediction.

        Args:
            input_size: Number of features per time step (default 1).
            hidden_size: LSTM hidden state dimension.
            num_layers: Number of stacked LSTM layers.
        """

        def __init__(self, input_size=1, hidden_size=16, num_layers=1):
            super().__init__()
            self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
            self.fc = nn.Linear(hidden_size, 1)

        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :])

    # Normalize data
    R_norm = (R_max_all - R_max_all.mean()) / R_max_all.std()

    # Sliding window: use 3 previous cycles to predict next
    window = 3
    X_train, y_train = [], []
    for i in range(len(R_norm) - window):
        X_train.append(R_norm[i:i + window])
        y_train.append(R_norm[i + window])
    X_train = torch.tensor(np.array(X_train), dtype=torch.float32).unsqueeze(-1)
    y_train = torch.tensor(np.array(y_train), dtype=torch.float32).unsqueeze(-1)

    torch.manual_seed(0)
    model = CycleLSTM(input_size=1, hidden_size=16, num_layers=1)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    losses = []
    for epoch in range(200):
        model.train()
        optimizer.zero_grad()
        pred = model(X_train)
        loss = criterion(pred, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Final training loss: {losses[-1]:.4f}")

    # Predict Cycle 25 using last 3 cycles (22, 23, 24)
    model.eval()
    with torch.no_grad():
        last_window = torch.tensor(R_norm[-window:], dtype=torch.float32).reshape(1, window, 1)
        pred_norm = model(last_window).item()

    pred_R25 = pred_norm * R_max_all.std() + R_max_all.mean()
    print(f"\nLSTM prediction for Cycle 25: {pred_R25:.1f}")
    print(f"(based on the previous {window} cycles)")

In [ ]:
if TORCH_OK:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(losses, color="steelblue")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("MSE Loss")
    axes[0].set_title("LSTM Training Loss / LSTM 훈련 손실")
    axes[0].grid(True, alpha=0.3)

    # Sequence predictions
    model.eval()
    preds_all = []
    with torch.no_grad():
        for i in range(len(R_norm) - window):
            x_in = torch.tensor(R_norm[i:i + window], dtype=torch.float32).reshape(1, window, 1)
            preds_all.append(model(x_in).item())
    preds_all = np.array(preds_all) * R_max_all.std() + R_max_all.mean()

    axes[1].plot(cycles_all, R_max_all, "o-", color="darkorange", label="Observed", markersize=8)
    axes[1].plot(cycles_all[window:], preds_all, "s-", color="steelblue",
                 label="LSTM predictions", markersize=6)
    axes[1].plot(25, pred_R25, "D", color="crimson", markersize=14, label="Cycle 25 forecast")
    axes[1].set_xlabel("Cycle number / 주기 번호")
    axes[1].set_ylabel("Peak amplitude R_max / 피크 진폭")
    axes[1].set_title("LSTM Cycle Predictions / LSTM 주기 예측")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6. Cycle 25 Ensemble Prediction Visualization / Cycle 25 앙상블 예측 시각화

### English
Here we visualize Table 2 from Petrovay (2020) — a summary of Cycle 25 predictions from different methodologies. The consensus range is ~100-140 (SSN v2), with dynamo-based predictions somewhat lower and neural networks predicting a very weak cycle.

### 한국어
Petrovay (2020)의 Table 2 — 다양한 방법론의 Cycle 25 예측 요약 — 을 시각화한다. 컨센서스 범위는 약 100-140 (SSN v2)이며, 다이나모 기반 예측은 다소 낮고 신경망은 매우 약한 주기를 예측한다.

In [ ]:
# Table 2 from Petrovay (2020): Cycle 25 forecasts
forecasts_t2 = [
    # (method, peak_amplitude, error, category)
    ("Internal precursor", 175, 24, "precursor"),
    ("Polar precursor (Table 1)", 117, 15, "precursor"),
    ("Polar precursor (SoDA)", 120, 39, "precursor"),
    ("Helicity", 117, 10, "precursor"),
    ("Rush-to-the-poles", 130, 20, "precursor"),
    ("SFT (Jiang)", 124, 31, "SFT"),
    ("AFT (Upton-Hathaway)", 110, 10, "SFT"),
    ("2x2D dynamo (Labonville)", 89, 22, "dynamo"),
    ("Truncated dynamo (Kitiashvili)", 90, 15, "dynamo"),
    ("Wavelet tree", 132, 15, "extrapolation"),
    ("Simplex projection", 103, 25, "extrapolation"),
    ("Simplex proj/time-delay", 154, 12, "extrapolation"),
    ("Neuro-fuzzy NN", 90.7, 8, "ML"),
    ("Spatiotemporal NN", 57, 17, "ML"),
]

cat_colors = {
    "precursor": "steelblue",
    "SFT": "forestgreen",
    "dynamo": "purple",
    "extrapolation": "goldenrod",
    "ML": "crimson",
}

fig, ax = plt.subplots(figsize=(12, 8))

names = [f[0] for f in forecasts_t2]
amps = np.array([f[1] for f in forecasts_t2])
errs = np.array([f[2] for f in forecasts_t2])
cats = [f[3] for f in forecasts_t2]
colors = [cat_colors[c] for c in cats]

y_pos = np.arange(len(names))
ax.errorbar(amps, y_pos, xerr=errs, fmt="o", capsize=5, color="black", zorder=2)
for i, (amp, err, col) in enumerate(zip(amps, errs, colors)):
    ax.plot(amp, i, "o", markersize=12, color=col, zorder=3)

# Cycle 24 reference line
ax.axvline(116, color="red", linestyle="--", alpha=0.7, label="Cycle 24 observed = 116")
ax.axvspan(116 * 0.8, 116 * 1.2, alpha=0.1, color="red", label="+/- 20% of Cycle 24")

ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Predicted Cycle 25 peak amplitude (SSN v2) / 예측 Cycle 25 피크 진폭")
ax.set_title("Cycle 25 Ensemble Forecasts (Petrovay 2020, Table 2)\n" +
             "Cycle 25 앙상블 예측")

# Legend for categories
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cat_colors[c], label=c) for c in cat_colors]
legend_elements.append(plt.Line2D([0], [0], color="red", linestyle="--",
                                    label="Cycle 24 = 116"))
ax.legend(handles=legend_elements, loc="lower right")
ax.grid(True, alpha=0.3, axis="x")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nSummary statistics:")
print(f"  Median: {np.median(amps):.0f}")
print(f"  Mean:   {np.mean(amps):.0f} +/- {np.std(amps):.0f}")
print(f"  Range:  {amps.min():.0f} - {amps.max():.0f}")
print(f"\nBy category (mean +/- std):")
for cat in cat_colors:
    idx = [i for i, c in enumerate(cats) if c == cat]
    a = amps[idx]
    print(f"  {cat:15s}: {np.mean(a):.0f} +/- {np.std(a):.0f} (n={len(a)})")

## 7. Summary / 요약

### English
This notebook reproduced the five main classes of solar cycle prediction methods reviewed by Petrovay (2020):

| Method | Our Cycle 25 prediction | Petrovay consensus |
|--------|------------------------|---------------------|
| Polar field precursor | ~98-128 | 117 +/- 15 |
| Waldmeier effect | (rise-time relation) | Not a direct predictor |
| AR(2) time-series | ~130-150 | Range: 100-140 |
| Truncated dynamo | (oscillator ensemble) | 89 +/- 22 (2x2D) |
| LSTM toy | Variable | Not well-calibrated |

**Key conclusions**:
1. The **polar field precursor is the gold standard**, with demonstrated skill across Cycles 21-24.
2. Most methods agree on a Cycle 25 amplitude similar to Cycle 24 (~100-140).
3. Dynamo-based predictions trend slightly lower (~89-90).
4. Two neural network predictions (57, 90.7) favor a very weak cycle — awaiting validation.

### 한국어
이 노트북은 Petrovay (2020)의 리뷰에서 다룬 다섯 가지 주요 태양 주기 예측 방법을 재현했다:

| 방법 | 우리의 Cycle 25 예측 | Petrovay 컨센서스 |
|------|---------------------|---------------------|
| 극자기장 선행지표 | ~98-128 | 117 +/- 15 |
| Waldmeier 효과 | (상승 시간 관계) | 직접 예측자 아님 |
| AR(2) 시계열 | ~130-150 | 범위: 100-140 |
| Truncated dynamo | (진동자 앙상블) | 89 +/- 22 (2x2D) |
| LSTM 장난감 | 가변 | 보정 불충분 |

**핵심 결론**:
1. **극자기장 선행지표가 황금 표준**이며, Cycle 21-24에 걸쳐 예측력 입증.
2. 대부분 방법이 Cycle 24와 유사한 Cycle 25 진폭 (~100-140)에 합의.
3. 다이나모 기반 예측은 약간 낮은 경향 (~89-90).
4. 두 신경망 예측 (57, 90.7)은 매우 약한 주기를 선호 — 검증 대기 중.

---

**References**:
- Petrovay, K. (2020). "Solar cycle prediction". *Living Reviews in Solar Physics*, 17:2. [DOI](https://doi.org/10.1007/s41116-020-0022-z)
- Table 1 and Table 2 data from Petrovay (2020)
- Sunspot Number v2.0 data from SIDC/WDC-SILSO, Royal Observatory of Belgium, Brussels